In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [4]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```',
  'doc_id': '5ca6890c1a'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CO

In [5]:
query = "I just dicovered the course. Can I still join it?"
query_vector = model.encode(query)
results = vs_index.search(
    query_vector, 
    filter_dict ={'course' : 'llm-zoomcamp'},
    num_results=5
)


In [7]:
query_vector

array([-1.41252894e-02, -6.54833540e-02, -1.43545959e-02,  1.14488900e-02,
        2.40780190e-02, -1.27901891e-02, -7.32685402e-02, -9.61943045e-02,
       -9.84018371e-02, -1.51901525e-02, -2.52913199e-02,  2.94465683e-02,
       -6.68568444e-03,  3.76016907e-02,  4.79045557e-03, -3.66061293e-02,
       -4.09698412e-02, -1.85662825e-02, -1.50630130e-02,  1.36819938e-02,
       -8.81847814e-02,  3.56369019e-02, -5.92480563e-02,  3.17947157e-02,
       -1.08762062e-03,  8.42964463e-03,  2.97390744e-02,  1.10909000e-01,
       -1.51449973e-02, -9.18046236e-02, -2.75576673e-02,  7.33194128e-02,
        1.14634149e-02,  3.33831981e-02,  7.52449734e-03,  8.49998370e-03,
        5.54478653e-02, -7.34425336e-02,  3.77649590e-02,  2.31612846e-02,
       -4.27186228e-02, -3.22272256e-03, -1.87463425e-02,  4.70649675e-02,
        2.44336426e-02,  8.99475589e-02, -7.55199268e-02, -7.97262862e-02,
        2.65376959e-02,  3.57001796e-02,  2.88876649e-02, -3.25988941e-02,
       -1.98841449e-02,  

In [6]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [9]:

from rag_helper import RAGBase

class RAGVector(RAGBase):

    #we need to pass an embedder to the RAGVector class so that it can compute the vector for the query
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    #we need to override the search method to compute the vector for the query and then search the index
    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [11]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [12]:
vs_index.close

<bound method VectorSearchIndex.close of <sqlitesearch.vector.index.VectorSearchIndex object at 0x75db7ad633e0>>